In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob
import colorsys

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
def log_fit(x, a, b):
    return a + b * np.log(x)

In [ ]:
files = glob.glob('../../data/regression_outputs/regmodels_spatial_self/Sampling_kcenter/*/*/Fuse/Token_Concat_spatial_self_Spatial/results.csv')
print(len(files))
df_list = []
for file in files:
    tmp_df = pd.read_csv(file)
    country = file.split('/')[-5]
    city = file.split('/')[-4]
    label_sdg_file = f"../../data/processed/0labels/{country}.csv"
    labels_sdg = pd.read_csv(label_sdg_file)
    tmp_df = pd.merge(tmp_df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    tmp_df['country'] = country
    tmp_df['city'] = city
    df_list.append(tmp_df)
df = pd.concat(df_list, ignore_index=True)
df['country_city'] = df['country'] + '_' + df['city']
df

In [ ]:
# df.drop(['target', 'k', 'ID', 'country', 'city','SDG'], axis=1, inplace=True)
df.drop(['target', 'ID', 'country', 'city','SDG'], axis=1, inplace=True)
grouped = df.groupby(['ratio', 'country_city']).mean().reset_index()
grouped

In [ ]:
city_list = grouped['country_city'].unique().tolist()
city_list

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# --- 1. Custom country color mapping ---
country_colors = {
    'China': '#C0392B',     # muted red
    'US': '#2C3E50',        # midnight blue
    'France': '#5DADE2',    # soft blue-gray
    'Brazil': '#F1C40F',    # mustard yellow
    'Nigeria': '#27AE60',   # forest green
    'Portugal': '#922B21',  # burgundy
    'Australia': '#D35400', # ochre (Australian outback)
}

# Fallback color for unexpected countries.
default_color = 'gray'

city_08_results = []
legend_elements = {} 

for city in city_list:
    city_df = grouped[grouped['country_city'] == city]

    x = city_df['ratio']
    y = city_df['all_r2']
    
    # --- 2. Resolve country -> color ---
    # Assume city format is "Country_City"
    country_name = city.split('_')[0]
    
    # Look up the color; fall back to gray if not in the dictionary.
    line_color = country_colors.get(country_name, default_color)
    
    try:
        params_r2, covariance = curve_fit(log_fit, x, y)
        
        x_08 = np.exp((0.8 - params_r2[0]) / params_r2[1]) 
        city_08_results.append((city, x_08))
        print(f'City {city} reaches R2=0.8 at ratio: {x_08:.4f}')
        
        x_fit = np.linspace(0, 1, 1000)
        y_fit_r2 = log_fit(x_fit, *params_r2)
        
        # --- 3. Plot ---
        line, = ax.plot(x_fit*100, y_fit_r2, 
                        color=line_color, 
                        alpha=0.6, 
                        linewidth=2)
        
        # Record legend entries (one per country).
        if country_name not in legend_elements:
            legend_elements[country_name] = line

        a, b = params_r2
        
    except Exception as e:
        print(f"Fit failed for {city}: {e}")

# --- Style ---
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)

ax.set_xlim(0, 100)
ax.set_ylim(0, 1)

ax.tick_params(axis='both', which='major', labelsize=16)
ax.axhline(y=0.8, color='gray', linestyle='--', linewidth=1.5)

ax.set_xlabel('Average surveyed areas (%)', fontsize=16)
ax.set_ylabel('Overall $R^2$', fontsize=16)

# --- 4. Add legend in custom order ---
# Sort legend entries alphabetically or in a specified order.
sorted_countries = sorted(legend_elements.keys())
sorted_lines = [legend_elements[c] for c in sorted_countries]

ax.legend(sorted_lines, sorted_countries, 
          loc='lower right', fontsize=12, frameon=False)

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig1_city_curve.svg', format='svg', bbox_inches='tight')
plt.show()

In [ ]:
city_08_df = pd.DataFrame(city_08_results, columns=['city', 'ratio_08'])
city_08_df